<a href="https://colab.research.google.com/github/author-sanjay/AirSafetyAI/blob/Data-Normalization/AirCraftAccidentDataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from google.colab import drive
from collections import defaultdict
import json
import pandas as pd
import re
import numpy as np
from datetime import datetime

# Data Collection

###### Data collection has been done already from airsafety db from year 2000 to 2025 resulting up to 6500+ recorded incidents found to train the model. Please note that this data is only being used for research purposes and model training

# Data Processing

✈️ Data Preprocessing & Normalization Notes
1. Data Cleaning



*   Find & delete duplicates → Remove duplicate entries for consistency.
*   Fix unknown/missing dates → Handle invalid or missing dates.
*   Normalize time values → Standardize time formats.
*   Merge Date & Time → Create a single DateTime column.
* Convert to UTC → Ensure uniform time reference across all records.
* Normalize flight hours → Standardize numeric flight time fields.



2. Dropping Irrelevant / Biased Fields

* Drop operator (Owner/Operator) → Not useful for prediction.

* Drop MSN & Registration → Identifiers, no predictive value.

* Drop raw DateTime after merging/UTC conversion → Avoid redundancy.

* Drop investigating agency info → Administrative, not predictive.

* Drop airports (Departure, Destination) → Route-level details handled separately in model logic.

* Drop Confidence rating & Location → Not meaningful for model training.

* Drop fatalities/occupants counts → Would bias model (seriousness ≠ death toll).

3. Normalizing Categories

* Aircraft Damage → Collapse variations into None, Minor, Substantial, Destroyed, Missing, Unknown.

* Phase → Normalize to Takeoff, Initial climb, En route, Approach, Landing, Unknown.

* Remove ground-only phases (Taxi, Pushback/Towing, Standing).

* Nature of Flight → Collapse into broader groups:

* Passenger (all types)

* Cargo/Ferry

* Military/Govt

* Training/Test/Calibration

* Special Ops (firefighting, agricultural, parachuting, ambulance, survey, patrol)

* Other/Unknown/Illegal

4. Narrative (Keep for NLP)

* Retain Narrative text → Used for NLP to extract accident cause/trigger (birdstrike, hydraulic failure, structural issue, etc.).

In [2]:
file_path = "/content/drive/MyDrive/accidents.json"

# Load your data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Dictionary to track occurrences
seen = defaultdict(list)

for idx, entry in enumerate(data):
    # Composite key: Date + Time + Registration + Location
    key = f"{entry.get('Date','')}_{entry.get('Time','')}_{entry.get('Registration','')}_{entry.get('Location','')}"
    seen[key].append(idx)

# Find duplicates (keys with more than 1 entry)
duplicates = {k: v for k, v in seen.items() if len(v) > 1}


In [3]:


file_path = "/content/drive/MyDrive/accidents.json"
cleaned_file_path = "/content/drive/MyDrive/accidents_cleaned.json"
csv_file_path = "/content/drive/MyDrive/accidents_cleaned.csv"

#Load JSON into pandas
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)



# Handle unk. date YYYY
def normalize_date(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    match = re.match(r"unk\. date (\d{4})", val, flags=re.IGNORECASE)
    if match:
        year = int(match.group(1))
        return f"{year}-12-31"

    try:
        return pd.to_datetime(val, errors="coerce").strftime("%Y-%m-%d")
    except Exception:
        return None

df["Date"] = df["Date"].apply(normalize_date)

def normalize_time(val):
    val = str(val).strip()
    if not val:
        return "12:00"
    val = val.replace("LT", "").strip()

    try:
        t = pd.to_datetime(val, format="%H:%M", errors="coerce")
        if pd.isna(t):
            return "12:00"
        return t.strftime("%H:%M")
    except:
        return "12:00"

df["Time"] = df["Time"].apply(normalize_time)


df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    errors="coerce"
)

df.drop(columns=["Date", "Time"], inplace=True)

df["Datetime"] = df["Datetime"].dt.tz_localize("UTC")

def normalize_category(cat):
    if pd.isna(cat):
        return "Incident"
    cat = str(cat).strip().lower()
    if cat == "accident":
        return "Accident"
    elif cat == "incident":
        return "Incident"
    elif cat == "serious incident":
        return "Serious Incident"
    elif cat == "unlawful interference":
        return "Unlawful Interference"
    elif cat == "uk":
        return "Unknown"
    elif cat == "other":
        return "Other"
    else:
        return "Incident"

df["Category"] = df["Category"].apply(normalize_category)

df.drop(columns=["Owner/operator"], inplace=True)

df.drop(columns=["Registration", "MSN"], inplace=True)

def clean_hours(x):
    if pd.isna(x) or str(x).strip() == "":
        return np.nan
    try:
        return float(str(x).split()[0].replace(",", ""))
    except:
        return np.nan

df["Total airframe hrs"] = df["Total airframe hrs"].apply(clean_hours)

def bucketize_hours(x):
    if pd.isna(x):
        return "Mid-life"
    elif x < 5000:
        return "New"
    elif x < 20000:
        return "Mid-life"
    else:
        return "Old"

df["airframe_bucket"] = df["Total airframe hrs"].apply(bucketize_hours)

df["Year of manufacture"] = pd.to_numeric(df["Year of manufacture"], errors="coerce")
df["aircraft_age"] = df["Datetime"].dt.year - df["Year of manufacture"]
df.loc[df["Year of manufacture"].isna(), "aircraft_age"] = np.nan
df.drop(columns=["Datetime"], inplace=True)
df.drop(columns=["Investigating agency"], inplace=True)
df.drop(columns=["Destination airport","Departure airport"], inplace=True)
df.drop(columns=["Confidence Rating","Location"], inplace=True)
df.drop(columns=["Fatalities","Other fatalities"], inplace=True)


phase_mapping = {
    "Approach": "Approach",
    "Landing": "Landing",
    "En route": "Cruise",
    "Taxi": "Ground",
    "Take off": "Takeoff",
    "Initial climb": "Takeoff",
    "Pushback / towing": "Ground",
    "Standing": "Ground",
    "Manoeuvring  (airshow, firefighting, ag.ops.)": "Special ops",
    "Unknown": "Unknown",
    "": "Unknown"
}

df["Phase"] = df["Phase"].map(lambda x: phase_mapping.get(str(x).strip(), "Unknown"))
df = df[df["Phase"] != "Ground"].reset_index(drop=True)


nature_mapping = {
    "Passenger - Scheduled": "Passenger",
    "Passenger - Non-Scheduled/charter/Air Taxi": "Passenger",
    "Passenger": "Passenger",
    "Executive": "Passenger",
    "Cargo": "Cargo",
    "Military": "Military",
    "Private": "Private",
    "Training": "Training",
    "Ferry/positioning": "Ferry/Positioning",
    "Fire fighting": "Special Operations",
    "Parachuting": "Special Operations",
    "Agricultural": "Special Operations",
    "Ambulance": "Special Operations",
    "Survey": "Special Operations",
    "Aerial patrol": "Special Operations",
    "Test": "Test/Demo",
    "Calibration/Inspection": "Test/Demo",
    "Demo/Airshow/Display": "Test/Demo",
    "Illegal Flight": "Illegal",
    "Unknown": "Unknown",
    "": "Unknown",
    "-": "Unknown",
    "SF": "Unknown"
}

df["Nature"] = df["Nature"].replace(nature_mapping)


# Extracting Incident Category Using NLP

### Source of Classification Data (ADREP / ECCAIRS)

The classification data we are using comes from internationally recognized aviation safety taxonomies:

* **ICAO / SKYbrary website**
Provides the official ADREP taxonomy (Accident/Incident Data Reporting) maintained by ICAO. This taxonomy defines standard categories and codes for classifying the causes, contributing factors, and types of aviation accidents and incidents.

*  **ECCAIRS (European Coordination Centre for Aviation Incident Reporting Systems)**
ECCAIRS publishes Data Definition Standards (DDS) that implement the ADREP taxonomy in structured form (attributes like Events, Occurrence Categories, Occurrence Classes, etc.). These standards are widely used by national investigation authorities and safety databases around the world.

In our dataset, we extracted ~96 distinct events from these standards. Each event includes:

1. ATA Code (system/component reference)

2. Event Name

3. Description

4. Broader Group (e.g., Flight Controls, Engine/Powerplant, Fire/Explosion)

5. Approach Category (our added field to help downstream airport-approach decision models)

### Why This Matters

We could have simply used an accidents.json file (with only past cases) and classified narratives against those. But that would give us pre-structured data limited to accidents that have already happened.

By instead using the official ICAO/ECCAIRS classification standards:

* We cover all known types of aviation events (not just those present in our dataset).

* We can train or fine-tune models to recognize potential causes of incidents, even those that have not yet occurred in the historical record.

* We align with international safety reporting practices, so our outputs are compatible with ICAO, IATA, and national investigation bodies.

### Why We Need This Taxonomy

* It provides a hierarchical, standardized vocabulary for aviation incidents.

* Enables NLP models to classify narratives into specific causes (e.g., “rudder malfunction”) and broader categories (e.g., “System/Component Failure – Non-Powerplant”).

* Supports predictive analysis: training models not just on what happened before, but on what can happen based on aviation safety knowledge.

In [4]:
import json

with open("/content/adrep_events.json", "r", encoding="utf-8") as f:
    adrep_events = json.load(f)

# Use "name" or "short form" as candidate labels
candidate_causes = [event["name"] for event in adrep_events]

print("Number of candidate labels:", len(candidate_causes))
print("Sample:", candidate_causes[:10])


Number of candidate labels: 96
Sample: ['Aircraft performance related event', 'Degraded performance', 'Towing and taxiing equipment related event', 'Towing related event', 'Parking and mooring related event', 'Placards and Markings related event', 'Servicing related event', 'Air conditioning and pressurization system related event', 'Autoflight system related event', 'Communication systems related event']


Zero Shot Test to check classifier

In [6]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

narrative = "On the approach to Tokyo, the pilot noticed a slight tilt in the aircraft probably caused by malfunctioning of rudder."

result = classifier(narrative, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result["labels"][:3], result["scores"][:3]):
    print(f"{label}: {score:.4f}")


Device set to use cuda:0


Aircraft rudder related event: 0.2850
Aircraft flight control related event: 0.1424
Aircraft empennage stucture related event: 0.0764


In [8]:
best_label = result["labels"][0]

matched_event = next((event for event in adrep_events if event["name"] == best_label), None)

if matched_event:
    print("Cause:", matched_event["name"])
    print("ATA Code:", matched_event["ata_code"])
    print("Approach:", matched_event["approach"])
    print("Group:", matched_event["broad_group"])


Cause: Aircraft rudder related event
ATA Code: 5540
Approach: APPROACH_STRAIGHT
Group: Flight Controls


In [10]:
narrative2 = "During climb, the pilot reported that the rudder was jammed and aircraft could only maintain straight flight."

result2 = classifier(narrative2, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result2["labels"][:3], result2["scores"][:3]):
    print(f"{label}: {score:.4f}")


Aircraft rudder related event: 0.2012
Aircraft flight control related event: 0.1371
Aircraft performance related event: 0.0704


In [11]:
narrative3 = "While preparing for descent, the hydraulic system pressure dropped suddenly, leading to difficulty in operating the landing gear and brakes."


result3 = classifier(narrative3, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result3["labels"][:3], result3["scores"][:3]):
    print(f"{label}: {score:.4f}")


Landing gear related event: 0.1770
Hydraulic system related event: 0.1672
Pneumatic system related event: 0.0718


In [12]:
narrative4 = "Shortly after takeoff, the left engine suffered a flameout with smoke visible from the nacelle."


result4 = classifier(narrative4, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result4["labels"][:3], result4["scores"][:3]):
    print(f"{label}: {score:.4f}")


Engine indicating system related event: 0.0776
Flight compartment related event: 0.0756
Aircraft performance related event: 0.0479


In [13]:
narrative5 = "On a long-haul flight, the crew declared emergency due to fuel leak in the left wing tank."
result5 = classifier(narrative5, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result5["labels"][:3], result5["scores"][:3]):
    print(f"{label}: {score:.4f}")


Fuel system related event: 0.1749
Fuel leak: 0.1669
Flight compartment related event: 0.1162


In [14]:
narrative6 = "Cabin crew reported smoke coming from the forward galley ovens, forcing the pilots to divert immediately."
result6 = classifier(narrative6, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result6["labels"][:3], result6["scores"][:3]):
    print(f"{label}: {score:.4f}")

Flight compartment related event: 0.2174
Buffet/galleys related event: 0.0737
Passenger compartment related event: 0.0695


In [15]:
best_label = result6["labels"][0]

matched_event = next((event for event in adrep_events if event["name"] == best_label), None)

if matched_event:
    print("Cause:", matched_event["name"])
    print("ATA Code:", matched_event["ata_code"])
    print("Approach:", matched_event["approach"])
    print("Group:", matched_event["broad_group"])


Cause: Flight compartment related event
ATA Code: 2510
Approach: APPROACH_ANY
Group: Other


In [16]:
narrative7 = "While on initial climb, the aircraft ingested a flock of birds into the right engine causing vibrations."
result7 = classifier(narrative7, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result7["labels"][:3], result7["scores"][:3]):
    print(f"{label}: {score:.4f}")

Engine indicating system related event: 0.0610
Flight compartment related event: 0.0557
Aircraft performance related event: 0.0547


In [17]:
narrative8 = "The aircraft descended below the minimum safe altitude and struck trees before initiating a go-around."
result8 = classifier(narrative8, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result8["labels"][:3], result8["scores"][:3]):
    print(f"{label}: {score:.4f}")

Aircraft flight control related event: 0.0826
Degraded performance: 0.0711
Aircraft performance related event: 0.0696


In [18]:
narrative9 = "On final approach, the crew discovered that the nose landing gear did not extend due to mechanical failure."
result9 = classifier(narrative9, candidate_causes, multi_label=False)  # multi_label allows multiple matches

# Show top 3 matches
for label, score in zip(result9["labels"][:3], result9["scores"][:3]):
    print(f"{label}: {score:.4f}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Landing gear related event: 0.3369
Aircraft empennage stucture related event: 0.0702
Landing gear position and warning system  related event: 0.0497
